# Trees & Graphs — Coding Interview Prep

BFS, DFS, binary trees, and graph traversal problems.

In [ ]:
from collections import deque
from typing import Optional, List

class TreeNode:
    def __init__(self, val=0, left=None, right=None):
        self.val = val
        self.left = left
        self.right = right

def build_tree(vals):
    """Build tree from level-order list. None = missing node."""
    if not vals: return None
    root = TreeNode(vals[0])
    q = deque([root])
    i = 1
    while q and i < len(vals):
        node = q.popleft()
        if i < len(vals) and vals[i] is not None:
            node.left = TreeNode(vals[i])
            q.append(node.left)
        i += 1
        if i < len(vals) and vals[i] is not None:
            node.right = TreeNode(vals[i])
            q.append(node.right)
        i += 1
    return root

print("TreeNode and helpers defined")

## 1. Tree Traversals

In [ ]:
def inorder(root):    # Left → Root → Right (sorted order for BST)
    return inorder(root.left) + [root.val] + inorder(root.right) if root else []

def preorder(root):   # Root → Left → Right
    return [root.val] + preorder(root.left) + preorder(root.right) if root else []

def postorder(root):  # Left → Right → Root
    return postorder(root.left) + postorder(root.right) + [root.val] if root else []

def level_order(root):  # BFS
    if not root: return []
    result, q = [], deque([root])
    while q:
        level = []
        for _ in range(len(q)):
            node = q.popleft()
            level.append(node.val)
            if node.left: q.append(node.left)
            if node.right: q.append(node.right)
        result.append(level)
    return result

#     3
#    / \
#   9  20
#     / \
#    15   7
root = build_tree([3, 9, 20, None, None, 15, 7])
print(f"Inorder:     {inorder(root)}")     # [9, 3, 15, 20, 7]
print(f"Preorder:    {preorder(root)}")    # [3, 9, 20, 15, 7]
print(f"Postorder:   {postorder(root)}")   # [9, 15, 7, 20, 3]
print(f"Level order: {level_order(root)}") # [[3], [9, 20], [15, 7]]

## 2. Binary Tree Problems

In [ ]:
# Maximum depth
def max_depth(root):
    return 1 + max(max_depth(root.left), max_depth(root.right)) if root else 0

# Symmetric tree
def is_symmetric(root):
    def mirror(l, r):
        if not l and not r: return True
        if not l or not r: return False
        return l.val == r.val and mirror(l.left, r.right) and mirror(l.right, r.left)
    return mirror(root.left, root.right) if root else True

# Path sum (root to leaf)
def has_path_sum(root, target):
    if not root: return False
    if not root.left and not root.right: return root.val == target
    return has_path_sum(root.left, target - root.val) or has_path_sum(root.right, target - root.val)

# Lowest Common Ancestor
def lca(root, p, q):
    if not root or root.val == p or root.val == q:
        return root.val if root else None
    left = lca(root.left, p, q)
    right = lca(root.right, p, q)
    return root.val if left and right else left or right

# Max path sum (any path)
def max_path_sum(root):
    best = [float('-inf')]
    def dfs(node):
        if not node: return 0
        l = max(dfs(node.left), 0)
        r = max(dfs(node.right), 0)
        best[0] = max(best[0], node.val + l + r)
        return node.val + max(l, r)
    dfs(root)
    return best[0]

root = build_tree([3, 9, 20, None, None, 15, 7])
print(f"Max depth: {max_depth(root)}")    # 3
print(f"Symmetric: {is_symmetric(build_tree([1,2,2,3,4,4,3]))}")  # True
print(f"Path sum 38: {has_path_sum(root, 38)}")  # True (3→20→15)
print(f"Max path sum: {max_path_sum(root)}")  # 42 (15+20+7)

## 3. Binary Search Tree

In [ ]:
def is_valid_bst(root, lo=float('-inf'), hi=float('inf')):
    if not root: return True
    if root.val <= lo or root.val >= hi: return False
    return is_valid_bst(root.left, lo, root.val) and is_valid_bst(root.right, root.val, hi)

def kth_smallest_bst(root, k):
    stack, n = [], 0
    curr = root
    while stack or curr:
        while curr:
            stack.append(curr); curr = curr.left
        curr = stack.pop()
        n += 1
        if n == k: return curr.val
        curr = curr.right

bst = build_tree([5, 3, 6, 2, 4, None, None, 1])
print(f"Valid BST: {is_valid_bst(bst)}")      # True
print(f"3rd smallest: {kth_smallest_bst(bst, 3)}")  # 3

## 4. Graph — BFS & DFS

In [ ]:
# Graph as adjacency list
graph = {
    0: [1, 2],
    1: [0, 3, 4],
    2: [0, 5],
    3: [1],
    4: [1, 5],
    5: [2, 4]
}

def bfs(graph, start):
    visited, q = {start}, deque([start])
    order = []
    while q:
        node = q.popleft()
        order.append(node)
        for neighbor in graph[node]:
            if neighbor not in visited:
                visited.add(neighbor)
                q.append(neighbor)
    return order

def dfs(graph, start, visited=None):
    if visited is None: visited = set()
    visited.add(start)
    order = [start]
    for neighbor in graph[start]:
        if neighbor not in visited:
            order.extend(dfs(graph, neighbor, visited))
    return order

def shortest_path_bfs(graph, start, end):
    q = deque([(start, [start])])
    visited = {start}
    while q:
        node, path = q.popleft()
        if node == end: return path
        for neighbor in graph[node]:
            if neighbor not in visited:
                visited.add(neighbor)
                q.append((neighbor, path + [neighbor]))
    return []

print(f"BFS from 0: {bfs(graph, 0)}")
print(f"DFS from 0: {dfs(graph, 0)}")
print(f"Shortest 0→5: {shortest_path_bfs(graph, 0, 5)}")

## 5. Graph Problems

In [ ]:
# Number of Islands (grid DFS)
def num_islands(grid):
    if not grid: return 0
    rows, cols = len(grid), len(grid[0])
    count = 0

    def dfs(r, c):
        if r < 0 or r >= rows or c < 0 or c >= cols or grid[r][c] != '1': return
        grid[r][c] = '#'  # mark visited
        for dr, dc in [(0,1),(0,-1),(1,0),(-1,0)]:
            dfs(r+dr, c+dc)

    for r in range(rows):
        for c in range(cols):
            if grid[r][c] == '1':
                dfs(r, c)
                count += 1
    return count

grid = [["1","1","0","0"],["1","0","0","1"],["0","0","0","1"]]
assert num_islands([row[:] for row in grid]) == 2

# Cycle detection (undirected)
def has_cycle(n, edges):
    adj = {i: [] for i in range(n)}
    for u, v in edges:
        adj[u].append(v); adj[v].append(u)

    visited = set()
    def dfs(node, parent):
        visited.add(node)
        for neighbor in adj[node]:
            if neighbor not in visited:
                if dfs(neighbor, node): return True
            elif neighbor != parent: return True
        return False

    for i in range(n):
        if i not in visited and dfs(i, -1): return True
    return False

assert has_cycle(4, [(0,1),(1,2),(2,3),(3,1)]) == True
assert has_cycle(4, [(0,1),(1,2),(2,3)]) == False

# Topological sort (Kahn's algorithm)
def topo_sort(n, prerequisites):
    adj = {i: [] for i in range(n)}
    in_degree = [0] * n
    for course, pre in prerequisites:
        adj[pre].append(course)
        in_degree[course] += 1
    q = deque([i for i in range(n) if in_degree[i] == 0])
    order = []
    while q:
        node = q.popleft()
        order.append(node)
        for neighbor in adj[node]:
            in_degree[neighbor] -= 1
            if in_degree[neighbor] == 0: q.append(neighbor)
    return order if len(order) == n else []  # empty = cycle exists

print(f"Topo sort: {topo_sort(4, [(1,0),(2,0),(3,1),(3,2)])}")  # [0,1,2,3] or [0,2,1,3]
print("Graph problems: all tests passed")

## Key Patterns

| Problem Type | Algorithm | Time | Tip |
|---|---|---|---|
| Tree traversal | DFS recursive | O(n) | Inorder=sorted for BST |
| Level order | BFS (deque) | O(n) | Group by queue size |
| Shortest path (unweighted) | BFS | O(V+E) | Always BFS for shortest |
| Connected components | DFS/Union-Find | O(V+E) | Mark visited in-place |
| Cycle detection | DFS with parent | O(V+E) | Track parent for undirected |
| Ordering with dependencies | Topological sort | O(V+E) | Use in-degree + BFS |